[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/cyclides/cyclides.ipynb)

# Dupin Cyclides on the 3-Sphere

This example ray traces Dupin cyclides in a curved space, the 3-sphere: the unit sphere in four dimensions, seen from inside, where light travels along great circles and there is no point at infinity. The starting point is the spherical cylinder, a tube around a great circle and straight along its core. A conformal dilation bends that core into a small circle, and the tube becomes a torus; aimed off the torus's centre, the dilation makes a lopsided Dupin cyclide; and the same dilation closes a spherical cone into a spindle with two conical points.

The 3-sphere is modelled in its conformal geometric algebra, where a point is a null antivector, a sphere a vector, and a quadric a map from points to their polar spheres, `Quadric = Sphere <- Point`, built as a sum of dyads. Every motion, a rotation of the 3-sphere or a dilation, moves a quadric on its input and its output alike, `motion >> quadric(motion << Point)`. A ray is the eye turned along a great circle; in the tangent of the half angle it is a parabola of points, and put into the quadric's form it gives a quartic whose coefficients are maps in the ray's direction, built once for every pixel before any pixel is seen.

In classical conformal geometry these motions are the Möbius transformations of $S^3$, and a Dupin cyclide is the image of a torus, a circular cylinder or a circular cone under one of them.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import CGA3D
from examples.util.animation import save_figure, show_animation
from examples.quadrics.cyclides import render, scenarios
from examples.quadrics.cyclides.core import pixel_grid, nearest_angle

# The conformal model of S³: the four directions x, y, z, w of R⁴, plus one direction e that squares to -1.
# A point of S³ is a unit direction of R⁴, an antivector, added to the centre of R⁴, mv.e.dual(): the direction
# squares to -1 and the centre to 1, and the point, their sum, to 0: a null antivector.
ga = CGA3D
ctx = NumpyContext(ga)
mv = ctx.multivector

Scalar = ga.gatype.scalar()                   # Scalar+
# A sphere is a vector, and the points on it are those with `sphere & point == 0`.
Sphere = ga.gatype.vector()                   # Vector+
Point = ga.gatype.antivector()                # Antivector-
# A direction at the eye, which sits at w, is an antivector in the duals of x, y and z; in the flat chart it is a
# displacement from the origin.
Direction = Point.derive.containing("w e")    # Antivector-
# A quadric maps a point to its polar sphere; the point is on the surface when it lies on its own polar. Motors are
# the rotations of R⁴, the dilations, and their products: the conformal motions of S³.
Quadric = ga.gatype((Sphere, Point))
Motor = ga.gatype.rotor()                     # Rotor+

# The eye, at w on S³, and the point opposite it, at -w.
origin = mv.w.dual() + mv.e.dual()            # [] Point-
antipode = mv.e.dual() - mv.w.dual()          # [] Point-

# The image: one unit direction per pixel, for a pinhole camera at the eye looking along -x with z up and a 90 degree
# view. A direction squares to minus its squared length.
shape = (180, 240)
pixel_right, pixel_up = pixel_grid(shape, np.radians(90))
rays = -mv.x.dual() + mv.y.dual() * pixel_right + mv.z.dual() * pixel_up              # [pixels] Direction-
pixels = rays / (rays | rays).abs().square_root()                              # [pixels] Direction-

np.set_printoptions(precision=6, suppress=True)

## 1. Points and spheres

A point of S³ is a unit direction of R⁴, carried as a null antivector: the direction with one unit of the centre `mv.e.dual()` added. A sphere of angular radius `r` about a point is the great sphere a quarter turn from it plus `cos(r)` units of `e`; at a quarter turn the `e` part vanishes, and the sphere is that great sphere. The regressive product `sphere & point` is zero exactly when the point lies on the sphere. `e & point` is the same for every point, and `e` itself is not a point of S³.

In [ ]:
# The angular radius of the sphere about the eye.
radius = 0.4

In [ ]:
# A sphere of angular radius r about the eye is the great sphere w, a quarter turn from it, plus e cos(r): against a
# point it gives the cosine of the point's angle from the eye less cos(r), which vanishes exactly when that angle is r.
sphere = mv.w + mv.e * np.cos(radius)                                        # [] Sphere+
# The eye turned by 0.4 towards x: the rotor exp(0.5 * angle * (x ^ w)) turns by `angle` in the xw plane.
turned_eye = (0.5 * radius * mv.xw).exp() >> origin                  # [] Point-

# The eye itself is 0 rad from the centre, not 0.4, so it is off the sphere; the turned eye is on it.
# Every point carries exactly one unit of e, so e & point is the same for all of them: e is not a point of S³.
incidence = stack((sphere & origin, sphere & turned_eye, mv.e & origin, mv.e & turned_eye))           # [incidences] Scalar+

print("sphere & eye, sphere & turned eye, e & eye, e & turned eye:", incidence.to_array())

## 2. Rays as great circles

Turning the eye towards a direction moves it along a great circle. The generator of that turn squares to a negative number, so its exponential sums to a cosine and a sine. The coefficient maps are commutators with the generator: the first and second derivatives of the turning eye.

In [ ]:
# The generator of the turn towards d: a rotation of R⁴ in the plane of the directions w and d, their join, which
# its dual generates. It squares to -1 for a unit d, and the factor 0.5 makes exp(t * ray_rotation(d)) turn the eye
# by exactly t.
ray_rotation = 0.5 * (mv.w.dual() & Direction).dual()                        # [] Bivector+ <- Direction-  
# Derivatives of the turning eye at t = 0: each is a commutator with the generator (the factor 2 undoes the
# commutator's 1/2). The first is the eye's velocity, heading off towards d; the second its acceleration.
# ray_rotation and ray_linear are each a map from a direction, so their commutator is a map from two, in order:
ray_linear = ray_rotation.commutator(origin) * 2                             # [] Point- <- Direction-
ray_quadratic = ray_rotation.commutator(ray_linear) * 2                      # [] Point- <- (Direction-, Direction-)
# Since the generator squares to -1, every higher derivative repeats these two up to sign, and the whole series
# sums to a cosine and a sine. The eye turned by t towards a unit d:
#     origin + sin(t) * ray_linear(d) + (1 - cos(t)) * ray_quadratic(d, d)

# Directions and points are both antivectors, so the printed type reads
# in antivector <- in antivector, in antivector: a point from two directions.
print(ray_quadratic)

In [ ]:
# Follow one unit direction to two angles, once with the exact rotor and once with the formula above, and test
# both results against the spheres of radius t about the eye.
direction = mv.y.dual() * 0.6 + mv.z.dual() * 0.8                            # [] Direction-
angles = np.array([0.7, 2.5])
exact = (ray_rotation(direction) * angles).exp() >> origin                   # [angles] Point-
circle = origin + ray_linear(direction) * np.sin(angles) + ray_quadratic(direction, direction) * (1 - np.cos(angles))  # [angles] Point-
# The spheres of radius t about the eye, and both points against them: all zero.
radius_spheres = mv.w + mv.e * np.cos(angles)                                # [angles] Sphere+
incidence = stack((radius_spheres & exact, radius_spheres & circle), axis=-1)   # [angles, methods] Scalar+

print("exact and circle points on the spheres of radius t:\n", incidence.to_array())

Points are only defined up to scale. Multiplied by `1 + u**2`, with the half angle `u = tan(t / 2)`, the circle becomes a parabola in `u`, and its bend is the antipode of the eye.

In [ ]:
# With u = tan(t / 2): sin(t) = 2u / (1 + u**2) and 1 - cos(t) = 2u**2 / (1 + u**2). Multiplying the turned eye by
# 1 + u**2, which does not move a point, clears the denominators:
#     (1 + u**2) * eye(t) == origin + 2u * ray_linear(d) + u**2 * (origin + 2 * ray_quadratic(d, d))
# The u**2 term is the bend. The origin in it carries the squared length of d, so the bend is quadratic in d like the
# rest of it; a direction, an antivector, squares to minus its squared length, so that length squared is -(d | d),
# one for a unit d. As u grows the bend dominates: t = pi reaches the antipode.
ray_bend = ray_quadratic * 2 - origin * (Direction | Direction)              # [] Point- <- (Direction-, Direction-)

In [ ]:
# The bend is the antipode, and the parabola, divided by 1 + u**2 again, lands on the circle.
# Five probes, the basis spheres, read every coefficient of a point: vanishing against all five means equal.
u = np.tan(angles / 2)
parabola = origin + ray_linear(direction) * (2 * u) + ray_bend(direction, direction) * u**2          # [angles] Point-
probes = stack((mv.x, mv.y, mv.z, mv.w, mv.e))                              # [probes] Sphere+
bend_gap = probes & (ray_bend(direction, direction) - antipode)             # [probes] Scalar+
parabola_gap = probes[:, None] & (parabola / (1 + u**2) - circle)           # [probes, angles] Scalar+

print("bend - antipode against five spheres:", bend_gap.to_array())
print("parabola / (1 + u**2) - circle:\n", parabola_gap.to_array())

## 3. Tracing the spherical cylinder

A quadric is a sum of dyads, one per sphere, with weights. As an extensor `Sphere <- Point` it sends a point to its polar sphere, and the point lies on the surface when it is incident with that polar. Each dyad contributes the squared incidence with its sphere.

The simplest one to trace is the spherical cylinder: all points at a fixed angle from a great circle, straight along its core. Two great spheres and `e` build it.

In [ ]:
# The cylinder's tube: its angle from the great circle at its core.
tube = 0.3

In [ ]:
# Each dyad s * (s & Point) sends a point to s, weighted by its incidence with s; paired with the point again it gives
# that incidence squared. For the point at p: z**2 + w**2 - sin(tube)**2, since e & point is the same unit for every point.
# The great circle in the xy plane is where z = w = 0, and z**2 + w**2 is sin**2 of the angle from it, so the zero set
# is every point at angle `tube` from that circle: a tube around a great circle, straight along its core.
cylinder = mv.z * (mv.z & Point) + mv.w * (mv.w & Point) - np.sin(tube)**2 * mv.e * (mv.e & Point)            # [] Sphere+ <- Point-

# Place it with rotations of R⁴, which are the rigid motions of S³: turning in yz stands its core upright in the
# image, and turning in xw carries it towards the view direction, to within 1.2 rad of the eye.
toward_view = (mv.xw * ((np.pi / 2 - 1.2) / 2)).exp()                       # [] Motor+
upright = (mv.yz * ((np.pi / 2 - 0.3) / 2)).exp()                            # [] Motor+
place = toward_view * upright                                                # [] Motor+
# A quadric moves as a sandwich on both sides: its input point is pulled back, its output sphere pushed forward.
placed_cylinder = (place >> cylinder(place << Point)).reshape(1)             # [surfaces] Sphere+ <- Point-

To trace, put each pixel's ray into the surface equation. The eye stays at `w`, so camera coordinates are world coordinates and the camera map is the identity. Along the parabola in `u` the equation is a polynomial of degree four in `u`, each coefficient a form in as many directions as its power.

`nearest_angle` solves for `u` and returns the smallest angle along the whole circle, so a ray that passes the antipode keeps going. At the hit, the polar sphere is the tangent sphere, and its length is the length of the surface gradient: the gradient along the ray's unit velocity over that length is the cosine between the ray and the normal. The function below does this for a batch of surfaces, and the later sections reuse it.

In index notation the $u^4$ coefficient reads as a symmetric tensor contracted four times with the direction, $Q_{ijkl}\, d^i d^j d^k d^l$.

In [ ]:
def trace(surfaces: Quadric, pixels: Direction) -> tuple[Scalar, np.ndarray]:
    """The headlight facing at the nearest hit, the cosine between the ray and the surface normal, and the angle t of
    that hit, per surface and pixel direction; t is inf on a miss."""
    # The quadric as a symmetric form on points: form(X, X) = X & Q(X), a point against its own polar sphere,
    # zero exactly on the surface. The eye stays at w, so the points are world points throughout.
    form = Point & surfaces                                                  # [surfaces] Scalar+ <- (Point-, Point-)

    # Put the ray X = origin + 2u ray_linear(d) + u**2 ray_bend(d, d) into form(X, X). A map passed as an argument keeps
    # its own arguments, so each term takes one direction per factor of d: ray_linear one, ray_bend two.
    # The form is symmetric, so the cross terms pair up, and each power of u collects its own form in d. These five
    # maps are the ray polynomial for every direction at once, built before any pixel is seen. The constant is the eye
    # against the surface.
    constant = form(origin, origin)                                          # [surfaces] Scalar+
    linear = 4 * form(origin, ray_linear)                                    # [surfaces] Scalar+ <- Direction-
    quadratic = 4 * form(ray_linear, ray_linear) + 2 * form(origin, ray_bend)   # [surfaces] Scalar+ <- (Direction-, Direction-)
    cubic = 4 * form(ray_linear, ray_bend)                                   # [surfaces] Scalar+ <- (Direction-, Direction-, Direction-)
    quartic = form(ray_bend, ray_bend)                                       # [surfaces] Scalar+ <- (Direction-, Direction-, Direction-, Direction-)
    # The u**4 coefficient is the first map here on four directions: ray_bend's two, twice. Nothing about four is
    # special: an extensor takes as many arguments as its expression has, each bound like any other, and
    # quartic(d, d, d, d) is the u**4 coefficient of the ray along d.

    # Tracing is now only binding: every pixel direction bound to every argument gives one quartic in u per surface and
    # pixel; [:, None] puts the surfaces on the first axis and the pixels on the second. nearest_angle turns every real
    # root u into the angle t = 2 atan(u) and keeps the smallest positive one along the whole circle, past the
    # antipode too, inf on a miss.
    angle = nearest_angle([
        constant[:, None],
        linear[:, None](pixels),
        quadratic[:, None](pixels, pixels),
        cubic[:, None](pixels, pixels, pixels),
        quartic[:, None](pixels, pixels, pixels, pixels)])                   # [surfaces, pixels]
    t = np.where(np.isfinite(angle), angle, 0.0)
    # The hit, and the ray's velocity there: the derivative in t of the turned eye, of unit speed.
    hit = origin + ray_linear(pixels) * np.sin(t) + ray_quadratic(pixels, pixels) * (1 - np.cos(t))   # [surfaces, pixels] Point-
    velocity = ray_linear(pixels) * np.cos(t) + ray_quadratic(pixels, pixels) * np.sin(t)          # [surfaces, pixels] Point-

    # At a point of the surface its polar sphere is the tangent sphere, and it grows with the surface's gradient.
    # The velocity against it, over its length, is the cosine between the ray and the normal: a headlight, 1 when
    # facing the eye.
    polar = surfaces[:, None](hit)                                           # [surfaces, pixels] Sphere+
    facing = -(polar & velocity) / (polar | polar).abs().square_root()       # [surfaces, pixels] Scalar+
    return facing, angle

facing, angle = trace(placed_cylinder, pixels)
save_figure(render.draw_facing(facing, angle, shape), "cyclides_cylinder");

## 4. Tori from a bent core

A dilation is a conformal map of S³ that keeps a point `toward` and its antipode fixed and slides every other point along its great circle through the two. Its generator is the dual of the diameter of R⁴ through them, the join `mv.e.dual() & toward` of the centre with the point. It acts on a quadric as a sandwich, on the output sphere and on the input point. Dilating towards `z`, a quarter turn from every point of the core, bends the core evenly into a smaller circle, and the cylinder becomes a torus. Seen from `z` a dilation is a uniform scaling, so it only sizes the torus; the tube sets its shape.

In [ ]:
# The tori's tubes about the core, and the strengths of their dilations, each chosen so that its torus fills the
# frame.
tubes = np.array([0.15, 0.5, 0.9])
strengths = np.array([1.55, 2.0, 2.4])

In [ ]:
def placement(toward: Point, ahead: float, tilt: float) -> Motor:
    """The motion turning a point `toward`, a unit direction of R⁴, onto the point `ahead` rad in front of the eye,
    after a tilt that keeps it fixed."""
    # The point `ahead` rad along the view direction -x.
    target = (mv.wx * (ahead / 2)).exp() >> mv.w.dual()
    # target * toward.inverse() turns `toward` onto `target` twice over; 1 plus it, normalized, is the rotor that turns
    # it straight onto `target`. The tilt turns in the yw plane, which leaves z and x alone, so for toward at z it only
    # spins the shape about its aim.
    onto_target = (1 + target * toward.inverse()).normalized()
    tilting = (mv.yw * (tilt / 2)).exp()
    return onto_target * tilting

# Tubes of 0.15, 0.5 and 0.9 rad about the core, all the same shape as the cylinder above but for their width:
tubes_around_core = (mv.z * (mv.z & Point) + mv.w * (mv.w & Point)
                     - np.sin(tubes)**2 * mv.e * (mv.e & Point))             # [tori] Sphere+ <- Point-
# The dilation towards z: its generator, the diameter through z read by its dual, is mv.ze. It squares to +1, so its exponential is hyperbolic, cosh and sinh, rather than a rotation. It keeps z
# and -z and pushes every other point along its great circle from -z towards z. The core, a quarter turn from z,
# shrinks into a small circle about z; each strength is chosen so that its torus fills the frame.
dilations = (mv.ze * (strengths / 2)).exp()                                  # [tori] Motor+
# Bring z, the torus's centre, into view; first dilate, then place.
views = placement(mv.z.dual(), 1.1, 0.8)                                     # [] Motor+
place = views * dilations                                                    # [tori] Motor+
tori = place >> tubes_around_core(place << Point)                            # [tori] Sphere+ <- Point-

facing, angle = trace(tori, pixels)
save_figure(render.draw_facing(facing, angle, shape), "cyclides_tori");

## 5. Dupin cyclides

Aiming the dilation at a point near the core breaks the spin symmetry. The tube is compressed on the side of the aim and swells on the other: the lopsided Dupin cyclides.

In [ ]:
# Per cyclide: its tube about the core, the strength of its dilation, and how far the dilation's aim leans from
# the torus's centre towards a point on the core.
tubes = np.array([0.25, 0.3, 0.2])
strengths = np.array([1.5, 1.2, 1.8])
leans = np.array([0.7, 0.8, 0.5])

In [ ]:
# The aim leans from z towards x, a point on the core: the dilation now pushes one side of the tube harder than the
# other. The aims are points of S³ as unit directions of R⁴.
aims = (mv.zx * (-leans / 2)).exp() >> mv.z.dual()                           # [cyclides] Point-
tubes_around_core = (mv.z * (mv.z & Point) + mv.w * (mv.w & Point)
                     - np.sin(tubes)**2 * mv.e * (mv.e & Point))             # [cyclides] Sphere+ <- Point-
dilations = ((mv.e.dual() & aims).dual() * (strengths / 2)).exp()           # [cyclides] Motor+
lopsided = dilations >> tubes_around_core(dilations << Point)                # [cyclides] Sphere+ <- Point-
# Each aim brought into view.
views = placement(aims, np.array([1.2, 1.3, 1.1]), np.array([-0.7, 0.6, 1.0]))   # [cyclides] Motor+
placed_lopsided = views >> lopsided(views << Point)                          # [cyclides] Sphere+ <- Point-

facing, angle = trace(placed_lopsided, pixels)
save_figure(render.draw_facing(facing, angle, shape), "cyclides_dupin");

## 6. Spindles from a cone

A spherical cone is the set of points whose geodesic from a vertex makes a fixed angle with an axis. It is built from two great spheres and `e`, like the cylinder. Its geodesics from the vertex meet again at the vertex's antipode, so the cone has two vertices. A strong dilation towards the axis point, a quarter turn from both vertices, pulls the vertices together and closes the cone into a spindle cyclide: two conical points joined by an outer and an inner sheet. Leaning the dilation off the axis bends the inner sheet into a banana, seen through an opening next to one vertex.

In [ ]:
# The cone's half-opening, and how far each dilation's aim leans from the axis point.
opening = 0.35
leans = np.array([0.0, 0.7])

In [ ]:
# A cone with vertex z, its axis towards x, half-opening 0.35 rad. Against the point at p the form reads
# x**2 + cos(opening)**2 * (z**2 - 1); on S³, 1 - z**2 = x**2 + y**2 + w**2, so the zero set is
# x**2 == cos(opening)**2 * (x**2 + y**2 + w**2): the geodesics leaving z at angle `opening` from the x axis.
# They all meet again at -z, the cone's second vertex.
cone = mv.x * (mv.x & Point) + np.cos(opening)**2 * (mv.z * (mv.z & Point) - mv.e * (mv.e & Point))            # [] Sphere+ <- Point-

# A strong dilation towards the axis point x, a quarter turn from both vertices, pulls the vertices towards each
# other; the second aim leans 0.7 rad towards y and bends the inner sheet. The aims are points of S³ as unit
# directions of R⁴.
aims = (mv.xy * (-leans / 2)).exp() >> mv.x.dual()                           # [spindles] Point-
dilations = ((mv.e.dual() & aims).dual() * (3.0 / 2)).exp()                  # [spindles] Motor+
spindles = dilations >> cone(dilations << Point)                             # [spindles] Sphere+ <- Point-

# Where the eye should aim: halfway between the two dilated vertices. The dilation keeps its aim and the aim's
# antipode, and slides the vertices z and -z, a quarter turn from both, alike along their great circles towards that
# antipode: it lies halfway between them.
midpoints = -aims                                                            # [spindles] Point-

In [ ]:
def orientation(angles: list[float]) -> Motor:
    """A rotation of R⁴ from six angles, one per coordinate plane, applied in turn."""
    rotor = mv.rotor()
    for plane, angle in zip((mv.xy, mv.xz, mv.xw, mv.yz, mv.yw, mv.zw), angles):
        rotor = rotor * (plane * angle).exp()
    return rotor

def view(centre: Point, angles: list[float], ahead: float, yaw: float, pitch: float) -> Motor:
    """The motion turning the shape about the origin of R⁴, carrying its centre, a unit direction of R⁴, to `ahead` rad
    in front of the eye, then turning the eye."""
    turn = orientation(angles)
    target = (mv.wx * (ahead / 2)).exp() >> mv.w.dual()
    # The turned centre straight onto the target.
    carry = (1 + target * (turn >> centre).inverse()).normalized()
    # Rotations in the xy and xz planes leave w, the eye, where it is: they turn the view, yaw and pitch.
    yawing = (mv.xy * (yaw / 2)).exp()
    pitching = (mv.xz * (pitch / 2)).exp()
    return yawing * pitching * carry * turn

# Five shots: which spindle, its orientation in the six planes, how far ahead, and the eye's yaw and pitch.
shots = [
    (0, [-0.186, -0.692, 2.658, 0.181, -0.282, -0.225], 1.10, -0.021, 0.061),
    (0, [1.124, 0.598, 0.155, 0.889, -0.164, -0.741], 1.05, -0.051, -0.062),
    (0, [0.464, 0.073, 0.536, -2.263, 0.817, -0.768], 1.05, -0.001, 0.060),
    (1, [-0.257, 0.18, 0.46, -0.999, -1.384, -0.004], 0.85, 0.137, -0.377),
    (1, [-0.511, -0.64, -0.64, 1.096, -1.168, -0.477], 0.85, -0.297, -0.358),
]
views = stack([view(midpoints[i], angles, ahead, yaw, pitch) for i, angles, ahead, yaw, pitch in shots])   # [shots] Motor+
shot_surfaces = stack([spindles[i] for i, *_ in shots])                     # [shots] Sphere+ <- Point-
placed_shots = views >> shot_surfaces(views << Point)                        # [shots] Sphere+ <- Point-

# Through the opening the eye sees both sides of the sheets, so the headlight takes the unsigned cosine.
facing, angle = trace(placed_shots, pixels)
save_figure(render.draw_facing(facing.abs(), angle, shape), "cyclides_spindles");

## 7. Symmetry instead of degree

The cylinder is built from great spheres and `e` only. That makes its polynomial in `u` palindromic: the `u**4` and constant coefficients agree, and the `u**3` and `u` coefficients are opposite. A palindromic quartic is a quadratic in `cot(t)`, and the four hits along a great circle come in antipodal pairs. The lopsided dilation breaks that symmetry.

The degree drops only for surfaces through the antipode of the eye, where the `u**4` coefficient vanishes.

In [ ]:
# The cylinder and a Dupin cyclide, both placed, as forms on points:
surfaces = stack((placed_cylinder[0], placed_lopsided[0]))                  # [surfaces] Sphere+ <- Point-
form = Point & surfaces                                                      # [surfaces] Scalar+ <- (Point-, Point-)

# A few pixel directions. The quartic in u is palindromic when its u**4 and constant coefficients agree and its
# u**3 and u coefficients are opposite; the two gaps below are zero exactly then. Swapping u for -1/u, which moves
# the point along the circle to its antipode, then maps roots to roots: the hits come in antipodal pairs.
sample = pixels[::20000]                                                     # [samples] Direction-
# The coefficients of u**4, u**3, u and the constant, as `trace` builds them, at the sample directions.
quartic = form(ray_bend, ray_bend)[:, None](sample, sample, sample, sample)    # [surfaces, samples] Scalar+
cubic = (4 * form(ray_linear, ray_bend))[:, None](sample, sample, sample)     # [surfaces, samples] Scalar+
linear = (4 * form(origin, ray_linear))[:, None](sample)                      # [surfaces, samples] Scalar+
constant = form(origin, origin)[:, None]                                     # [surfaces, 1] Scalar+
outer_gap = quartic - constant                                               # [surfaces, samples] Scalar+
odd_gap = cubic + linear                                                     # [surfaces, samples] Scalar+

print("u**4 - constant, for the cylinder and a Dupin cyclide:\n", outer_gap.to_array())
print("u**3 + u:\n", odd_gap.to_array())

## 8. A torus rolled around a circle

A circle is the meet of two spheres, a bivector, and its exponential turns points around the circle, the way a vortex ring carries fluid around its core. Around the torus's own core circle the flow only spins the tube in place and leaves the torus unchanged. Around that core tilted a little, one turn rolls and deforms the torus.

In [ ]:
# The frames of one turn around a circle, and the duration of each.
n_frames = 54
duration_ms = 40

In [ ]:
# A circle is the meet of two spheres, their wedge: the core of every torus above is z ^ w, the great circle where
# both great spheres z and w meet. Carry it with the middle torus's own placement, after a small tilt of 0.4 rad in
# the yw plane, so that it no longer lines up with the torus's core. The versors are unit, so the circle stays a
# unit circle: circle * circle == -1.
circle = (place[1] * (mv.yw * 0.2).exp()) >> mv.zw                  # [] Bivector+

# exp(circle * phase / 2) turns every point of S³ around that circle by `phase`, fixing the circle itself: points
# near it circle it quickly, like fluid around a vortex ring. One full turn over the frames, applied to the torus as
# a sandwich like any other motion; each frame is traced only when the animation asks for it.
phases = np.linspace(0.0, 2 * np.pi, n_frames, endpoint=False)
flow = (circle * (phases / 2)).exp()                                          # [frames] Motor+
rolling_torus = flow >> tori[1](flow << Point)                                # [frames] Sphere+ <- Point-
traced = (trace(surface.reshape(1), pixels) for surface in rolling_torus)

show_animation(render.facing_frames(traced, shape), "cyclides_vortex", duration_ms)

## 9. A linked vortex

The flat conformal model is one chart of S³, the stereographic projection from the antipode of the eye: the eye is its origin and the antipode its point at infinity. Flat constructions written with these two draw the same surfaces on S³. A hyperboloid of one sheet, inverted in a sphere outside it, closes its two ends in a single conical tip; a circle threading its opening turns it around, linked with a thin ring on that circle.

In [ ]:
# The flat chart about the eye, as the spheres of the flat conformal model: its origin, the sphere of radius zero about
# the eye, and its infinity, the one about the antipode.
chart_origin = (mv.w + mv.e) * 0.5                                           # [] Sphere+
chart_infinity = mv.e - mv.w                                                 # [] Sphere+
# The radius of the circle round the hyperboloid's waist, and the tube of the ring on it.
ring_radius, ring_tube = 2.5, 0.02

# A hyperboloid of one sheet, z**2 = x**2 + (y / 0.65)**2 - 1: a waist, and two ends that open towards infinity.
hyperboloid = (mv.z * (mv.z & Point) - mv.x * (mv.x & Point) - mv.y * (mv.y & Point) / 0.65**2
               + chart_infinity * (chart_infinity & Point))                  # [] Sphere+ <- Point-
# Inverted in the unit sphere about (1.7, 0, 0), outside it: infinity goes to that centre, so both ends meet there
# in a conical tip, and the waist becomes the rim of an opening. A translation of the chart is, on S³, a conformal
# map that keeps the antipode fixed, generated by the join of the antipode with the displacement; the inversion is a
# reflection in a sphere.
shift = ((antipode & (mv.x.dual() * 1.7)).dual() * (-1 / 2)).exp()           # [] Motor+
inversion = (shift >> (chart_origin - chart_infinity)).normalized()          # [] Sphere+
cyclide = inversion >> hyperboloid(inversion << Point)                       # [] Sphere+ <- Point-

# The circle about the z axis goes round the hyperboloid's waist; inverted with it, it threads the opening
# crosswise. Normalized before inverting, it stays a unit circle: circle * circle == -1.
sphere = chart_origin - chart_infinity * (ring_radius**2 / 2)                # [] Sphere+
circle = inversion >> (mv.z ^ sphere).normalized()                           # [] Bivector+
# The ring around it: the inverted sphere at unit weight, whose square is the circle's squared radius, dressed as a
# torus of the ring's tube in the plane z = 0.
sphere = inversion >> sphere
sphere = sphere / -(sphere | chart_infinity)
radius_squared = sphere.squared()
sphere = sphere - chart_infinity * (ring_tube**2 / 2)
ring = sphere * (sphere & Point) + radius_squared * (
    mv.z * (mv.z & Point) - ring_tube**2 * chart_infinity * (chart_infinity & Point))   # [] Sphere+ <- Point-

# The flat tracer's camera: its pose carries a camera at the origin, looking along -x with z up, to `position`
# looking at `target`. Moving the scene by the inverse brings that camera to the eye, which also looks along -x; the
# great circles through the eye are the chart's straight lines through its origin, so the view is the flat one.
position = mv.x.dual() * 2.8 - mv.y.dual() * 9.0 + mv.z.dual() * 4.8          # [] Direction-
target = mv.x.dual()                                                         # [] Direction-
sight = target - position                                                    # [] Direction-
forward = sight / (sight | sight).abs().square_root()                        # [] Direction-
# z, less its part along forward; the turn of -x onto forward, then of z onto up.
upright = mv.z.dual() - (mv.z.dual() | forward) * forward.inverse()           # [] Direction-
up = upright / (upright | upright).abs().square_root()                       # [] Direction-
aim = (1 + forward * (-mv.x.dual()).inverse()).normalized()                  # [] Motor+
roll = (1 + up * (aim >> mv.z.dual()).inverse()).normalized()                # [] Motor+
translation = ((antipode & position).dual() * (-1 / 2)).exp()                # [] Motor+
camera = (translation * roll * aim).inverse()                                # [] Motor+
# The linked vortex's image, larger than the others.
linked_shape = (315, 420)
narrow_right, narrow_up = pixel_grid(linked_shape, np.radians(44))
narrow_rays = -mv.x.dual() + mv.y.dual() * narrow_right + mv.z.dual() * narrow_up     # [pixels] Direction-
narrow = narrow_rays / (narrow_rays | narrow_rays).abs().square_root()       # [pixels] Direction-

# One turn around the circle over the frames. The ring stays, so it is traced once; per frame the cyclide is traced, and
# the image keeps the nearer hit in each pixel. The inversion centre is outside the solid, so the cyclide is lit on both
# sides.
flow = (circle * (np.linspace(0.0, 2 * np.pi, n_frames, endpoint=False) / 2)).exp()   # [frames] Motor+
ring_facing, ring_angle = trace((camera >> ring(camera << Point)).reshape(1), narrow)   # [1, pixels] Scalar+, [1, pixels]
turning = (trace((motion >> cyclide(motion << Point)).reshape(1), narrow) for motion in camera * flow)
traced = ((concatenate((facing, ring_facing)).abs(), np.concatenate((angle, ring_angle))) for facing, angle in turning)

show_animation(render.linked_frames(traced, linked_shape), "cyclides_linked_vortex", duration_ms)

## 10. The rest of the gallery

The other scenes of the flat tracer, each built in the chart about the eye and seen through its camera; `scenarios.py` holds their constructions.

In [ ]:
# One still per scene: rings, a peanut, pinched and two-lobed cyclides, and the six families of the sphere model.
still = ["cyclide", "peanut", "elliptic_ring", "split_ring", "pinched", "six_families", "hyperboloid", "two_lobed"]
images = [next(render.scene_frames(scenarios.flat_scene(name, 1), shape)) for name in still]
save_figure(render.draw_images(images, 4), "cyclides_gallery");

In [ ]:
# The scenes with vortices: each part turned once around its own circle.
for name in ["linked_tori", "pinched_vortex"]:
    show_animation(render.scene_frames(scenarios.flat_scene(name, n_frames), shape), f"cyclides_{name}", duration_ms)